# LLM-DM GPyTorch v2：全部运行，统一检查，一次确认

这是重新冻结的独立实验，不读取或覆盖 v1 已有结果。请使用新的 **GPU 运行时**。

点击「全部运行」后会自动：拉取固定版本 → 安装依赖（保留 CUDA PyTorch）→ 验证冻结数据/预算 → 挂载 Drive 并恢复 v2 备份 → 全部 19 个方法各跑完整 seed-0 pilot。
随后统一展示耗时、显存和训练诊断，**等待你输入一次确认**，再自动跑全部方法的正式 seeds 38–45。不用逐个修改方法或 seed。

Google Drive 授权仍需要你操作；确认 pilot 前正式训练不会启动。任一失败/中断/备份异常都会停止，不自动重试、不改预算。断线恢复的是已完成结果，不是训练中间参数。

默认主实验共 19 个 pilot + 152 个正式任务；选中 fixed-1B 后任务数量翻倍，需检查两种设置的 pilot。队列不保证能在一次 Colab 会话内跑完。不要同时打开两个 notebook 写同一备份目录。


In [ ]:
# 可选：同时跑 fixed-1B ablation；不需要改方法或 seed。
INCLUDE_FIXED_1B = False #@param {type:"boolean"}

from pathlib import Path
import importlib, json, os, platform, re, subprocess, sys

CODE_COMMIT = "9d70e1458239142353e34cc596ba2b8b8764871d"
ASSETS_COMMIT = "e1fb2c34c5dcd6ce9555afaf6b31c2c1a2cade80"
UPSTREAM_COMMIT = "37269969a0957448d51622e0c083977bc5d260e8"
ROOT = Path("/content/llmdm-gpytorch-v2")
REPO = ROOT / "code"
RELEASE_REPO = ROOT / "release"
UPSTREAM = ROOT / "data-recipes"
SETTINGS = ("multi_scale", "fixed_1b") if INCLUDE_FIXED_1B else ("multi_scale",)
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("请在 Google Colab GPU 运行时中打开此 notebook。")
if not all(re.fullmatch(r"[0-9a-f]{40}", revision) for revision in (CODE_COMMIT, ASSETS_COMMIT, UPSTREAM_COMMIT)):
    raise RuntimeError("发布版本尚未固定，禁止运行。")
if any(path.is_symlink() for path in (REPO, *REPO.parents)):
    raise RuntimeError("代码目录不能包含符号链接。")
if not REPO.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/Kaiyue2003/llm-design-bench.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "-c", "core.autocrlf=false",
                    "checkout", "--detach", CODE_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
if head != CODE_COMMIT or dirty:
    raise RuntimeError("代码版本不一致或已有修改；请用新运行时，不覆盖旧文件。")
for name in ("llm_design_bench", "colab_support", "colab_batch", "colab_v2"):
    loaded = sys.modules.get(name)
    if loaded is not None and not Path(getattr(loaded, "__file__", "")).resolve().is_relative_to(REPO.resolve()):
        raise RuntimeError(f"{name} 已从另一版本加载，请使用新的 Python 运行时。")
sys.path.insert(0, str(REPO / "scripts"))
from colab_v2 import checkout_exact, validate_release, install_environment, prepare_workflow
checkout_exact("https://github.com/Kaiyue2003/llm-design-bench.git", REPO, CODE_COMMIT)
checkout_exact("https://github.com/Kaiyue2003/llm-design-bench.git", RELEASE_REPO, ASSETS_COMMIT)
checkout_exact("https://github.com/namkoong-lab/data-recipes.git", UPSTREAM, UPSTREAM_COMMIT)
ASSETS = RELEASE_REPO / "experiments/llmdm_forward_gpytorch_v2"
release = validate_release(ASSETS, code_commit=CODE_COMMIT, upstream_commit=UPSTREAM_COMMIT)
print("新实验:", release["experiment_id"], "设置:", SETTINGS)
print("冻结计划:", release["plan_id"])


## 1. 安装与检查环境

GPyTorch 及相关依赖使用发布包的固定版本；保留 Colab 当前的 CUDA PyTorch。正常情况下安装后可以直接导入，不必手工处理 editable install。若本会话已经加载了被替换的依赖，会停止并要求重启 Python。未知依赖冲突不会被忽略。


In [ ]:
os.environ["MPLCONFIGDIR"] = str(ROOT / "matplotlib-cache")
install_environment(REPO, ASSETS)
from colab_v2 import runtime_identity
print(json.dumps(runtime_identity(), indent=2))


## 2. 授权 Drive，校验数据并恢复 v2

请完成 Google 弹窗授权。数据校验应为 454 logged / 184 main visible / 26 fixed-1B；不重新切分。状态与备份都按新 plan_id 隔离，不会读取 v1。

Drive 保存追加式、带校验和的状态归档。备份失败就停；不会删除旧归档。请确保有足够空间。运行代码会具有所授权的 Drive 访问权限，只运行你信任的 notebook。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
workflow = prepare_workflow(REPO, ASSETS, UPSTREAM, release)
batch = workflow.batch
STATE, BACKUPS = batch.state, batch.backups
import pandas as pd
from IPython.display import display
print("校验通过：454 logged / 184 main visible / 26 fixed-1B / 19 methods")
print("本地状态:", STATE)
print("Drive 备份:", BACKUPS)
display(pd.DataFrame(batch.preview(settings=SETTINGS, phase="pilot")))


## 3. 自动跑完整 pilot → 统一确认 → 自动跑全部 8 seeds

这个单元会启动训练。每次只运行一个任务；每个方法使用冻结预算，不是缩小版 smoke test。CPU/GPU、float32/float64 按方法策略自动选择。

pilot 完成后会显示全部成本与诊断，并出现输入框。检查每个方法的数值稳定性、训练摘要、耗时和显存，再输入屏幕显示的 `RUN FORMAL …`。不根据 pilot oracle 分数挑方法、改预算或挑 seed。

输入其他内容则保留 pilot 并停止，正式训练不启动。再次运行会校验并跳过已完成的相同任务；只有未曾尝试的任务会启动，中断或失败的任务必须先人工检查。


In [ ]:
def show_pilot_report(rows):
    table = pd.DataFrame(rows)
    columns = ["setting", "run_id", "status", "train_size", "candidate_budget",
               "device", "dtype", "method_seconds", "peak_gpu_memory_mib",
               "unique_candidate_count", "artifact_checks"]
    display(table[[column for column in columns if column in table]])
    for row in rows:
        print(f"\n{row['setting']} / {row['run_id']}")
        print(json.dumps({key: row.get(key) for key in ("training_summary", "diagnostics")},
                         ensure_ascii=False, indent=2))

run_result = workflow.run_all(settings=SETTINGS, display_report=show_pilot_report)


## 4. 验证正式覆盖率与结果位置

每个方法/设置必须有 8 个 verified_successes、0 pending、0 blocked 才完成正式覆盖。这不是方法效果排名；逐候选、逐 seed 的完整结果保存在 formal 目录及 Drive 校验归档内。

如果训练异常，队列停止处的错误会给出具体方法/seed。不要通过删除失败记录、改 seed、忽略环境差异或直接重跑来隐藏问题；先检查日志与备份。此 notebook 不提供保活或自动失败重试。


In [ ]:
coverage = pd.DataFrame(workflow.coverage(settings=SETTINGS))
display(coverage)
complete = bool(len(coverage)) and bool(((coverage.verified_successes == 8) &
             (coverage.pending == 0) & (coverage.blocked == 0)).all())
print("所选范围正式实验已完成。" if complete else "正式覆盖尚未完成；不要把部分结果当作最终表。")
print("逐候选/逐 seed 结果:", STATE / "formal")
print("Drive 备份:", BACKUPS)
